In [1]:
## Summarization notebook

In [2]:
# Load environment variables and set up auto-reload
import dotenv
import os
from dotenv import load_dotenv
dotenv.load_dotenv("../env_workshop")



%load_ext autoreload
%autoreload 2

In [3]:
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL")
PHOENIX_PROJECT_NAME=os.environ.get("PHOENIX_PROJECT_NAME")


from phoenix.otel import register
from openinference.instrumentation import using_metadata
from openinference.instrumentation.langchain import LangChainInstrumentor
from opentelemetry import trace
from opentelemetry.trace import NoOpTracerProvider


if os.environ.get("PHOENIX_COLLECTOR_ENDPOINT"):
    # configure the Phoenix tracer
    tracer_provider = register(
    project_name=PHOENIX_PROJECT_NAME, 
    auto_instrument=False 
    )
    LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
else:
    dummy_tracer_provider = NoOpTracerProvider()

tracer = trace.get_tracer(__name__)

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: npatta01
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: llm-tracing.np-training.dev:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [4]:
from utils import show_prompt
from datetime import datetime
from langgraph.graph import StateGraph, START, END
import rich


In [5]:
import operator
from typing_extensions import TypedDict, Annotated, List, Sequence
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages

In [6]:
from langchain_openai import ChatOpenAI


In [7]:
writer_model = ChatOpenAI(
    model="gpt-4.1",
    base_url=OPENAI_BASE_URL,
    max_tokens=32000
) # model="anthropic:claude-sonnet-4-20250514", max_tokens=64000

In [8]:
class ResearcherState(TypedDict):
    """
    State for the research agent containing message history and research metadata.
    
    This state tracks the researcher's conversation, iteration count for limiting
    tool calls, the research topic being investigated, compressed findings,
    and raw research notes for detailed analysis.
    """
    notes: str
    research_brief:str
    final_report:str
    messages: Annotated[Sequence[BaseMessage], add_messages]



In [9]:
final_report_generation_prompt = """Based on all the research conducted, create a comprehensive, well-structured answer to the overall research brief:
<Research Brief>
{research_brief}
</Research Brief>

CRITICAL: Make sure the answer is written in the same language as the human messages!
For example, if the user's messages are in English, then MAKE SURE you write your response in English. If the user's messages are in Chinese, then MAKE SURE you write your entire response in Chinese.
This is critical. The user will only understand the answer if it is written in the same language as their input message.

Today's date is {date}.

Here are the findings from the research that you conducted:
<Findings>
{findings}
</Findings>

Please create a detailed answer to the overall research brief that:
1. Is well-organized with proper headings (# for title, ## for sections, ### for subsections)
2. Includes specific facts and insights from the research
3. References relevant sources using [Title](URL) format
4. Provides a balanced, thorough analysis. Be as comprehensive as possible, and include all information that is relevant to the overall research question. People are using you for deep research and will expect detailed, comprehensive answers.
5. Includes a "Sources" section at the end with all referenced links

You can structure your report in a number of different ways. Here are some examples:

To answer a question that asks you to compare two things, you might structure your report like this:
1/ intro
2/ overview of topic A
3/ overview of topic B
4/ comparison between A and B
5/ conclusion

To answer a question that asks you to return a list of things, you might only need a single section which is the entire list.
1/ list of things or table of things
Or, you could choose to make each item in the list a separate section in the report. When asked for lists, you don't need an introduction or conclusion.
1/ item 1
2/ item 2
3/ item 3

To answer a question that asks you to summarize a topic, give a report, or give an overview, you might structure your report like this:
1/ overview of topic
2/ concept 1
3/ concept 2
4/ concept 3
5/ conclusion

If you think you can answer the question with a single section, you can do that too!
1/ answer

REMEMBER: Section is a VERY fluid and loose concept. You can structure your report however you think is best, including in ways that are not listed above!
Make sure that your sections are cohesive, and make sense for the reader.

For each section of the report, do the following:
- Use simple, clear language
- Use ## for section title (Markdown format) for each section of the report
- Do NOT ever refer to yourself as the writer of the report. This should be a professional report without any self-referential language. 
- Do not say what you are doing in the report. Just write the report without any commentary from yourself.
- Each section should be as long as necessary to deeply answer the question with the information you have gathered. It is expected that sections will be fairly long and verbose. You are writing a deep research report, and users will expect a thorough answer.
- Use bullet points to list out information when appropriate, but by default, write in paragraph form.

REMEMBER:
The brief and research may be in English, but you need to translate this information to the right language when writing the final answer.
Make sure the final answer report is in the SAME language as the human messages in the message history.

Format the report in clear markdown with proper structure and include source references where appropriate.

<Citation Rules>
- Assign each unique URL a single citation number in your text
- End with ### Sources that lists each source with corresponding numbers
- IMPORTANT: Number sources sequentially without gaps (1,2,3,4...) in the final list regardless of which sources you choose
- Each source should be a separate line item in a list, so that in markdown it is rendered as a list.
- Example format:
  [1] Source Title: URL
  [2] Source Title: URL
- Citations are extremely important. Make sure to include these, and pay a lot of attention to getting these right. Users will often use these citations to look into more information.
</Citation Rules>
"""

In [10]:
def get_today_str() -> str:
    """Get current date in a human-readable format."""
    return datetime.now().strftime("%a %b %-d, %Y")


def final_report_generation(state: ResearcherState):
    """
    Final report generation node.

    Synthesizes all research findings into a comprehensive final report
    """

    findings = state.get("notes", [])


    final_report_prompt = final_report_generation_prompt.format(
        research_brief=state.get("research_brief", ""),
        findings=findings,
        date=get_today_str()
    )

    final_report = writer_model.invoke([HumanMessage(content=final_report_prompt)])

    return {
        "final_report": final_report.content, 
        "messages": ["Here is the final report: " + final_report.content],
    }

# ===== GRAPH CONSTRUCTION =====
# Build the overall workflow
deep_researcher_builder = StateGraph(ResearcherState)

# Add workflow nodes
deep_researcher_builder.add_node("report", final_report_generation)
deep_researcher_builder.add_edge(START, "report") 
deep_researcher_builder.add_edge("report", END)

# Compile the full workflow
agent = deep_researcher_builder.compile()

In [11]:
notes = """
List of Queries and Tool Calls Made                                                                                

 1 Searched for: "best coffee shops Seattle ambiance interior design atmosphere comfort vibe 2025"                 
 2 Searched for: "top Seattle coffee shops ambiance reviews interior design atmosphere comfort 2025 site:yelp.com  
   OR site:tripadvisor.com OR site:seattletimes.com OR site:seattlemet.com"                                        
 3 Searched for: "Storyville Coffee Company ambiance interior design comfort atmosphere reviews"                   
 4 Searched for: "Café Hagen Seattle coffee ambiance vibe interior design reviews"                                 
 5 Searched for: "Santo Coffee Seattle ambiance atmosphere interior design reviews"                                
 6 Searched for: "Sugar Bakery Seattle coffee shop ambiance interior design reviews"                               

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Fully Comprehensive Findings                                                                                       

 • Seattle is more than just the birthplace of Starbucks—it's one of the best cities in the U.S. for coffee lovers,
   digital nomads, and aesthetic café seekers.                                                                     
   This 2025 Seattle coffee guide features handpicked local cafés known for specialty coffee, cozy workspaces, and 
   beautifully designed interiors.                                                                                 
   Olympia Coffee blends vibrant ambiance with exceptional quality, making it a comfortable and photogenic         
   workspace.                                                                                                      
   Starbucks Reserve Roastery is part museum, part café, and open late for those who need a caffeine fix past      
   sundown.                                                                                                        
   Root (Ballard) is a coffee shop meets plant store—literally, one of the most Instagrammable spots in town.      
   Cafés with Instagram-worthy vibes such as Olympia Coffee, Caffè Umbria, 203° Fahrenheit Coffee Company, and Café
   Hagen are also detailed.                                                                                        
   Additional recommendations cover spots like Anchorhead Coffee, Root in Ballard with its plant store atmosphere, 
   Cardoon (Ballard), Espresso Vivace and Kaladi Brothers in Capitol Hill, Zoka Coffee in Greenlake, and Café      
   Allegro in the University District.                                                                             
   The guide includes descriptions of café interiors, specialties, and operating hours, serving travelers and      
   locals seeking quality coffee experiences and cozy environments ideal for work or relaxation [1].               
 • Top 10 Best Coffee Shops With Ambience Near Seattle, Washington include:                                        

 1 Storyville Coffee Company                                                                                       
 2 Sugar Bakery [2].                                                                                               

 • Top 10 Best Aesthetic Cafe Near Seattle, Washington include:                                                    

 1 Santo Coffee (4.7/5, 206 reviews, 3.6 mi, 1325 NE 65th St, Seattle, WA 98115)                                   
 2 Café Hagen (4.4/5, 563 reviews) [4].                                                                            

 • Located in Seattle, Wet Clay Cafe stands out for its creative ambiance and engaging activities. It's not just a 
   café; it's a space where... [3].                                                                                
 • The coffee at Storyville Coffee Pike cafe is anomalous. The cafe itself has a nice cosy atmosphere. I ordered an
   oat vanilla latte which cost $13.30 !!! Given the great reviews (and the cost) I was expecting an excellent     
   coffee but tbh it was just average [7].                                                                         
 • Storyville Coffee Company is a privately-owned specialty coffee business based in Seattle, founded in 2006. With
   51-200 employees and approximately 949 followers on LinkedIn, Storyville operates four coffee shops in Seattle  
   and ships freshly roasted coffee nationwide. The company's mission is to end human trafficking, aligning its    
   specialty coffee offerings—such as espresso, baked goods, pastries, breakfast items, and coffee                 
   subscriptions—with this social cause. Key staff include Jeremy Barwood (Supply Chain Manager), Carla Persinger  
   (People Operations), and Ashley Feith (Barista). Storyville has not publicly disclosed funding rounds or        
   investors. The company's locations are 9459 Coppertop Loop, 94 Pike Street #34, 2128 Queen Anne Ave N, and 1001 
   1st Ave—all in Seattle, Washington. Similar companies in food and beverage services include Dough Joy and       
   Messenger Coffee Company. Storyville Coffee Company is a privately owned, Seattle-based specialty coffee company
   with a mission to end human trafficking. Storyville has been shipping exquisitely roasted fresh coffee all      
   around the US since 2006. Our four Seattle shops demonstrate our passion for beauty and excellence. Specialties 
   include Coffee, Espresso, Baked Goods, Pastries, Breakfast, Coffee Subscription, and Bakery [8].                
 • Cafe Hagen is the perfect brunch spot with the most beautiful interior and delicious food and coffee. It's been 
   on my to-go list for so long [10].                                                                              
 • Café Hagen boasts a modern decor with elements like wood furniture and Viking—however, some reviews indicated   
   that the café can be noisy at times. Ambiance: Café Hagen boasts a modern decor with elements like wood         
   furniture and Viking— [11].                                                                                     
 • Cafe Hagen, Seattle: See 12 unbiased reviews of Cafe Hagen, rated 4.2 of 5 on Tripadvisor and ranked #797 of    
   2541 restaurants in Seattle [12].                                                                               
 • Santo Coffee, located in the Roosevelt neighborhood at 1325 NE 65th St, Seattle, is a startup coffee shop owned 
   by a family passionate about quality, craftsmanship, and community. After a long wait since its planned opening 
   in Fall 2018, Santo Coffee finally opened, offering a meticulously designed, upscale atmosphere with features   
   like floor-to-ceiling windows, black marble counters, and unique window bar seating. One of the owners is former
   Seattle Sounders star Fredy Montero. They serve Devocion coffee, a New York-based roaster known for its unique  
   farm-to-cup process that delivers coffee within 10 to 30 days of harvest, setting them apart from other Seattle 
   cafes. Pastries are provided by Sémillon Bakery in Capitol Hill. The reviewer tried a cortado and raspberry     
   brioche and praised the atmosphere, service, and overall experience, rating it highly. Santo Coffee stands out  
   as a sophisticated space within a quaint neighborhood and is ideal for visitors seeking a high-end coffee       
   experience. The café has quickly become a local favorite and distinguishes itself with both its coffee selection
   and design. Santo Coffee is now open and you can tell great care and meticulous detail was put into the design  
   of the shop. One of the owners of Santo Coffee is former Seattle Sounders star, Fredy Montero. Their coffee is  
   from Devocion, a roaster based in New York which roasts beans from Colombia within 10 to 30 days of harvest     
   versus the industry standard of 6 months. Santo Coffee is easily the “fanciest” space in Seattle with features  
   like floor-to-ceiling windows, black marble counters, and a bar that runs along the window that becomes a bench 
   [13].                                                                                                           
 • The ambience was lovely as well. It wasn't too loud but the atmosphere was still welcoming. I got an iced panela
   latte which was absolutely fantastic. I asked [15].                                                             
 • Sugar Bakery: Absolutely loved this coffee shop! The decor is so cute inside and they have a bunch of art and   
   stuff on display. It was nice and toasty in there and lots of [17].                                             
 • Sugar Bakery & Cafe, Seattle: See 60 unbiased reviews of Sugar Bakery & Cafe, rated 4.3 of 5 on Tripadvisor and 
   ranked #472 of 2547 restaurants in Seattle [18].                                                                

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
List of All Relevant Sources (with citations in the report)                                                        

[1] Seattle Coffee Guide 2025: The Best Cafes for Work, Tourists ...                                               
https://www.junky-traveler.com/post/seattle-coffee-guide-2025-the-best-cafes-for-work-tourists-instagram-lovers    

[2] TOP 10 BEST Coffee Shops With Ambience in Seattle, WA - Yelp                                                   
https://www.yelp.com/search?find_desc=Coffee+Shops+With+Ambience&find_loc=Seattle%2C+WA                            

[3] Unique Cafes with Ambiance in Seattle - Lemon8-app                                                             
https://www.lemon8-app.com/experience/seattle-cafe-with-unique-ambiance?region=us                                  

[4] TOP 10 BEST Aesthetic Cafe in Seattle, WA - Updated 2025                                                       
https://www.yelp.com/search?find_desc=Aesthetic+Cafe&find_loc=Seattle%2C+WA                                        

[5] THE 10 BEST Cafés in Seattle (Updated 2025)                                                                    
https://www.tripadvisor.ca/Restaurants-g60878-c8-Seattle_Washington.html                                           

[6] Storyville Coffee Company ambiance interior design comfort atmosphere reviews                                  
https://www.tripadvisor.com/Restaurant_Review-g60878-d4439485-Reviews-Storyville_Coffee_Pike_Place-Seattle_Washingt
on.html                                                                                                            

[7] Storyville Coffee Company | LinkedIn                                                                           
https://www.linkedin.com/company/storyville-coffee-company                                                         

[8] Storyville Coffee Company - Crunchbase Company Profile & Funding                                               
https://www.crunchbase.com/organization/storyville-coffee-company                                                  

[9] Cafe Hagen Review Seattle - TikTok                                                                             
https://www.tiktok.com/discover/cafe-hagen-review-seattle                                                          

[10] Café Hagen Modern All-Day Cafe - Reviews, Photos & Phone ...                                                  
https://caf-hagen-modern-all-day-cafe.wheree.com/                                                                  

[11] CAFE HAGEN, Seattle - Restaurant Reviews, Photos & Phone Number                                               
https://www.tripadvisor.com/Restaurant_Review-g60878-d24004712-Reviews-Cafe_Hagen-Seattle_Washington.html          

[12] Santo Coffee - Roosevelt - Seattle Unexplored                                                                 
https://seattleunexplored.com/santo-coffee-roosevelt-seattle-review/                                               

[13] Santo Coffee Co, Seattle, WA - Reviews, Ratings, Tips and Why You ...                                         
https://wanderlog.com/place/details/438102/santo-coffee-co                                                         

[14] Santo Coffee - Yelp                                                                                           
https://www.yelp.com/biz/santo-coffee-seattle                                                                      

[15] Sugar Bakery, Seattle, WA - Reviews, Ratings, Tips and ... - Wanderlog                                        
https://wanderlog.com/place/details/431675/sugar-bakery                                                            

[16] SUGAR BAKERY - Updated November 2025 - 609 Photos - Yelp                                                      
https://www.yelp.com/biz/sugar-bakery-seattle-3                                                                    

[17] Sugar Bakery & Cafe - Seattle Restaurants - Tripadvisor                                                       
https://www.tripadvisor.in/Restaurant_Review-g60878-d2687559-Reviews-Sugar_Bakery_Cafe-Seattle_Washington.html     

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Every substantive and relevant detail from all discovered sources about Seattle’s best coffee shops for ambiance   
has been preserved verbatim and organized above, along with all source citations.                                  


"""



research_brief = 'I want to identify the best coffee shop in Seattle, with a primary focus on coffee quality and atmosphere, as these are my top priorities. Please research and compare Seattle coffee shops based on the quality of their coffee (considering factors such as bean sourcing, roasting methods, brewing techniques, and taste) and the overall atmosphere (including ambiance, decor, comfort, and vibe). I have not specified a particular neighborhood, so consider the entire city of Seattle. Other factors such as price, food offerings, or popularity should only be considered if they are relevant to quality or atmosphere, but are not primary concerns unless they directly impact those two dimensions. Please prioritize information from official coffee shop websites, reputable review platforms (such as Google Reviews or Yelp), and respected local publications. If possible, provide direct links to these sources.'


In [12]:
state = ResearcherState({
    "research_brief": research_brief,   
    "notes": notes,
})

In [13]:
with tracer.start_as_current_span("summarization"):
    result = agent.invoke(state)

In [14]:
result

{'notes': '\nList of Queries and Tool Calls Made                                                                                \n\n 1 Searched for: "best coffee shops Seattle ambiance interior design atmosphere comfort vibe 2025"                 \n 2 Searched for: "top Seattle coffee shops ambiance reviews interior design atmosphere comfort 2025 site:yelp.com  \n   OR site:tripadvisor.com OR site:seattletimes.com OR site:seattlemet.com"                                        \n 3 Searched for: "Storyville Coffee Company ambiance interior design comfort atmosphere reviews"                   \n 4 Searched for: "Café Hagen Seattle coffee ambiance vibe interior design reviews"                                 \n 5 Searched for: "Santo Coffee Seattle ambiance atmosphere interior design reviews"                                \n 6 Searched for: "Sugar Bakery Seattle coffee shop ambiance interior design reviews"                               \n\n───────────────────────────────────────────────

In [15]:
rich.print(result['final_report'])

# The Best Coffee Shop in Seattle for Coffee Quality and Atmosphere: A Comprehensive Analysis

## Introduction

Seattle is known as a world-renowned coffee destination, boasting a rich array of independent cafés celebrated for 
their dedication to quality and creativity. For those prioritizing specialty coffee and exceptional atmosphere, 
discerning the “best” coffee shop in Seattle involves a nuanced comparison of numerous standouts citywide. Detailed
below is a comparison and in-depth analysis of Seattle’s leading coffee shops, focusing exclusively on coffee 
quality—including sourcing, roasting, and brewing methods—as well as the overall ambiance, decor, comfort, and 
vibe.

## Defining Excellence: Criteria

In evaluating Seattle’s coffee shops, two dimensions are prioritized:
- **Coffee Quality**: This encompasses the sourcing of beans, roasting philosophy, brewing precision, and the 
resulting taste.
- **Atmosphere**: This includes physical ambiance, decor, comfort, aesthetic details, noise level, and the overall 
customer vibe.

Other factors, such as food offerings or popularity, are only considered where they support or detract from these 
core priorities.

## Leading Contenders: Detailed Profiles

### Olympia Coffee

Olympia Coffee (with multiple Seattle locations) is frequently named among the city’s top cafés for both quality 
and atmosphere.

- **Coffee Quality**: Olympia Coffee is lauded for its meticulous direct sourcing and roasting practices, 
delivering vibrant and freshly roasted artisanal cups. Their approach emphasizes traceability, ethical 
relationships, and a hands-on roasting process that yields flavorful espresso and filter options.
- **Atmosphere**: The shops are inviting, aesthetically pleasing, and feature open, well-designed interiors. 
Olympia’s spaces provide comfort for both those seeking a workspace and those wishing to relax, creating a 
photogenic yet unpretentious environment.
- **Distinction**: Olympia stands out as a go-to for both coffee connoisseurs and those seeking a comfortable yet 
lively meeting spot[1].

### Storyville Coffee Company

Storyville Coffee, with several prominent locations including downtown and Pike Place, is recognized for both its 
social mission and ambiance.

- **Coffee Quality**: Storyville focuses on specialty coffee, roasting and serving fresh beans with an emphasis on 
consistency. However, consumer impressions have occasionally found the taste to be more solid than extraordinary, 
especially relative to its premium pricing. Reviewers consider the drinks well-crafted but occasionally 
underwhelming in the context of high expectations[2].
- **Atmosphere**: Storyville offers a warm, cozy, and intimate environment. The spaces are carefully designed, 
leaning into comfort and calm, making them popular for extended stays. The mission-driven aspect (supporting 
anti-trafficking initiatives) further enhances the sense of purpose and care[3].
- **Distinction**: The brand’s commitment to beauty in both the cup and environment consistently earns accolades 
for ambiance regardless of personal taste preferences[2][3].

### Café Hagen

Café Hagen, situated conveniently near Lake Union, is celebrated as much for its modern Scandinavian design as for 
its high-quality drinks.

- **Coffee Quality**: Café Hagen boasts a diverse specialty menu, with careful bean selection and sophisticated 
drinks. Reviews consistently describe the coffee as delicious and freshly prepared, suitable for both casual 
drinkers and aficionados.
- **Atmosphere**: Modern, bright, and stylish, Café Hagen’s space incorporates natural woods, minimalist elements, 
and ample natural light. Some reviewers have noted higher noise levels at peak times, but the interior remains an 
archetype of “aesthetic café” design[4][5].
- **Distinction**: Café Hagen doubles as a brunch destination, making it ideal for social visits and those seeking 
a lively, Instagram-worthy vibe[5][6].

### Santo Coffee

Santo Coffee,